# Lab | Simple LLM App with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

### Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.


In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3"  "langchain-openai<0.2"

<br>

### LangSmith

[LangSmith](https://smith.langchain.com) is a platform built by the LangChain team for debugging, testing, and monitoring LLM applications. It works alongside LangChain (though it can also be used independently) and lets you inspect every step of a chain or agent's execution.

Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls.
As these applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent. This is where LangSmith becomes especially useful: it automatically traces each run of your chain, showing you the exact inputs and outputs of every step (prompt formatting, model calls, parsing, etc.), how long each step took, and how many tokens were used. This makes it much easier to debug unexpected outputs, catch errors, and understand the cost/performance of your application.

Before you can use LangSmith, you'll need to set it up:

1. **Create a LangSmith account**: Go to [smith.langchain.com](https://smith.langchain.com) and sign up for a free account (you can sign in with GitHub, Google, or email).
2. **Create an API key**: Once logged in, go to *Settings* (or your organization's settings) and generate a new API key under the "API Keys" section.
3. **Set the environment variables**: In the next cell, you'll be prompted to enter that API key, which will be stored as the `LANGCHAIN_API_KEY` environment variable. We also set `LANGCHAIN_TRACING_V2` to `"true"` to turn on tracing so that every run of your chain gets logged and can be inspected in the LangSmith UI.

Note: LangSmith is optional — your chain will still work without it — but enabling it lets you see detailed traces of each step in your chain, which is very useful for debugging.

In [ ]:
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()


In [ ]:
# For this exercise, we'll use also OpenAI

os.environ["OPENAI_API_KEY"] = getpass.getpass()

In [ ]:
# | output: false
# | echo: false

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo")


Let's first use the model directly. `ChatModel`s are instances of LangChain "Runnables", which means they expose a standard interface for interacting with them. To just simply call the model, we can pass in a list of messages to the `.invoke` method.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English into Italian"), # Feel free to try something else here!
    HumanMessage(content="My tailor is rich"),
]

model.invoke(messages)


If you've enabled LangSmith, this run has just been logged as a trace. We'll take a closer look at your traces at the end of this notebook.

## OutputParsers

Notice that the response from the model is an `AIMessage`. This contains a string response along with other metadata about the response. Oftentimes we may just want to work with the string response. We can parse out just this response by using a simple output parser.

We first import the simple output parser.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

One way to use it is to use it by itself. For example, we could save the result of the language model call and then pass it to the parser.

In [ ]:
result = model.invoke(messages)

In [ ]:
parser.invoke(result)

More commonly, we can "chain" the model with this output parser. This means this output parser will get called everytime in this chain. This chain takes on the input type of the language model (string or list of message) and returns the output type of the output parser (string).

We can easily create the chain using the `|` operator. The `|` operator is used in LangChain to combine two elements together.

In [ ]:
chain = model | parser

In [ ]:
chain.invoke(messages)

This chain now has two steps: first the language model is called, then the result is passed to the output parser. If you check LangSmith later, you'll be able to see both steps logged separately for this run.

## Prompt Templates

Right now we are passing a list of messages directly into the language model. Where does this list of messages come from? Usually, it is constructed from a combination of user input and application logic. This application logic usually takes the raw user input and transforms it into a list of messages ready to pass to the language model. Common transformations include adding a system message or formatting a template with the user input.

PromptTemplates are a concept in LangChain designed to assist with this transformation. They take in raw user input and return data (a prompt) that is ready to pass into a language model. 

Let's create a PromptTemplate here. It will take in two user variables:

- `language`: The language to translate text into
- `text`: The text to translate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

First, let's create a string that we will format to be the system message:

In [ ]:
system_template = "Translate the following into {language}:"

Next, we can create the PromptTemplate. This will be a combination of the `system_template` as well as a simpler template for where the put the text

In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

The input to this prompt template is a dictionary. We can play around with this prompt template by itself to see what it does by itself

In [ ]:
result = prompt_template.invoke(
    {
        "language": "italian",
        "text": "Good morning, how are you?"
    }
)

result

We can see that it returns a `ChatPromptValue` that consists of two messages. If we want to access the messages directly we do:

In [ ]:
result.to_messages()

## Chaining together components with LCEL

We can now combine this with the model and the output parser from above using the pipe (`|`) operator:

In [ ]:
chain = prompt_template | model | parser

In [ ]:
chain.invoke(
    {
        "language": "italian",
        "text": "Good morning, how are you?"
    }
)

This is a simple example of using [LangChain Expression Language (LCEL)](/docs/concepts/#langchain-expression-language-lcel) to chain together LangChain modules. There are several benefits to this approach, including optimized streaming and tracing support. This chain has three components (prompt template, model, and parser), and all three will show up as separate steps in LangSmith.

<br>


## Serving with LangServe

Now that we've built an application, we need to serve it. That's where LangServe comes in.
LangServe helps developers deploy LangChain chains as a REST API. You do not need to use LangServe to use LangChain, but in this guide we'll show how you can deploy your app with LangServe.

While the first part of this guide was intended to be run in a Jupyter Notebook or script, we will now move out of that. We will be creating a Python file and then interacting with it from the command line.

Install with:
```bash
pip install "langserve[all]"
```

### Server

To create a server for our application we'll make a `serve.py` file. This will contain our logic for serving our application. It consists of three things:
1. The definition of our chain that we just built above
2. Our FastAPI app
3. A definition of a route from which to serve the chain, which is done with `langserve.add_routes`


```python
#!/usr/bin/env python
from typing import List

from fastapi import FastAPI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langserve import add_routes

# 1. Create prompt template
system_template = "Translate the following into {language}:"
prompt_template = ChatPromptTemplate.from_messages([
    ('system', system_template),
    ('user', '{text}')
])

# 2. Create model
model = ChatOpenAI()

# 3. Create parser
parser = StrOutputParser()

# 4. Create chain
chain = prompt_template | model | parser


# 4. App definition
app = FastAPI(
  title="LangChain Server",
  version="1.0",
  description="A simple API server using LangChain's Runnable interfaces",
)

# 5. Adding chain route

add_routes(
    app,
    chain,
    path="/chain",
)

if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="localhost", port=8000)
```

And that's it! If we execute this file:
```bash
python serve.py
```
we should see our chain being served at [http://localhost:8000](http://localhost:8000).

<br>

### Playground

Every LangServe service comes with a simple [built-in UI](https://github.com/langchain-ai/langserve/blob/main/README.md#playground) for configuring and invoking the application with streaming output and visibility into intermediate steps.
Head to [http://localhost:8000/chain/playground/](http://localhost:8000/chain/playground/) to try it out! Pass in the same inputs as before - `{"language": "italian", "text": "hi"}` - and it should respond same as before.

### Client

Now let's set up a client for programmatically interacting with our service. We can easily do this with the `[langserve.RemoteRunnable](/docs/langserve/#client)`.
Using this, we can interact with the served chain as if it were running client-side.

In [ ]:
from langserve import RemoteRunnable

remote_chain = RemoteRunnable("http://localhost:8000/chain/")
remote_chain.invoke({"language": "italian", "text": None})

To learn more about the many other features of LangServe [head here](/docs/langserve).

<br>

## Explore your traces in LangSmith

Throughout this notebook, every call to `model.invoke(...)` and `chain.invoke(...)` was logged as a trace in LangSmith (if you completed the setup at the beginning). Now it's time to go inspect what was actually captured.

Go to [smith.langchain.com](https://smith.langchain.com), open your project, and complete the following tasks:

1. **Find your earliest run** (from `model.invoke(messages)`, right after loading the model). Open it and check the exact prompt that was sent to the model, and how many tokens were used.
2. **Find the run from `chain = model | parser`**. Notice that, unlike the first run, this trace has two steps. Click into each step to see what the model returned versus what the parser returned.
3. **Find the run from the final chain** (`prompt_template | model | parser`). This trace should show three steps. Compare the input of the first step with the input of the second step — how did the `ChatPromptTemplate` transform your input?
4. **Compare latencies**: pick any two runs and compare how long each step took. Which step is typically the slowest?

Write a short summary (2-3 sentences) of what you found in the cell below.

*Your summary here:*

<br>

## Conclusion

In this lab you practiced building an LLM application using LangChain Expression Language (LCEL) — chaining a prompt template, a chat model, and an output parser together with the `|` operator. You also used LangSmith to inspect exactly what happens inside a chain, and saw how to serve a chain as a REST API with LangServe.

For further reading on the core concepts of LangChain, we've got detailed [Conceptual Guides](/docs/concepts).

If you have more specific questions on these concepts, check out the following sections of the how-to guides:

- [LangChain Expression Language (LCEL)](/docs/how_to/#langchain-expression-language-lcel)
- [Prompt templates](/docs/how_to/#prompt-templates)
- [Chat models](/docs/how_to/#chat-models)
- [Output parsers](/docs/how_to/#output-parsers)
- [LangServe](/docs/langserve/)

And the LangSmith docs:

- [LangSmith](https://docs.smith.langchain.com)